# Verifying the quirks and the fixes

Run the cells top to bottom. Two kinds of cells:

* **✓ checks** run by themselves and print `✓` or `✗` per item; the last cell
  sums them up. A `✗` is a regression.
* **👆 by hand** show an editor with steps to follow: what should happen is
  written under **Expect**. These are the things only a person can check —
  clicks, keys, a finger.

The first cell makes this notebook use *this checkout* (`../src` and the
add-ons in `../addons`), not an installed copy of `sympy_editor`.

In [ ]:
import sys, os, json, tempfile, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "examples" else pathlib.Path.cwd()
for p in [ROOT / "src"] + sorted((ROOT / "addons").glob("sympy_editor_*")):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
for name in [m for m in sys.modules if m.startswith("sympy_editor")]:
    del sys.modules[name]                      # a copy imported earlier would shadow this checkout

import sympy
from sympy import *
import sympy_editor
from sympy_editor import Document
from sympy_editor.widget import SympyEditorWidget

print("sympy_editor from", sympy_editor.__file__)
print("SymPy", sympy.__version__)

RESULTS = []
def check(what, ok, detail=""):
    RESULTS.append((what, bool(ok)))
    print(("✓ " if ok else "✗ ") + what + (f"   [{detail}]" if detail and not ok else ""))

def raises(fn, *kinds):
    try:
        fn()
    except kinds or Exception as exc:        # noqa: B030
        return exc
    return None

x, y, z, t, w = symbols("x y z t w")
SCRATCH = pathlib.Path(tempfile.mkdtemp(prefix="sympy-editor-verify-"))
print("scratch folder:", SCRATCH)

## 1. Saved files are read, never run

A `.sympy` file can arrive by mail or from a file manager, and the apps open
it. Nothing in it may run: the steps are read by walking their syntax tree —
SymPy's constructors and literals only. Text you type in the editor, and a
string your own Python hands to `Document(...)`, keep the old (evaluating)
reading.

In [ ]:
marker = SCRATCH / "i-ran"
evil_step = f"(__import__('pathlib').Path({str(marker)!r}).write_text('owned'), Symbol('x'))[1]"
evil_file = json.dumps({"sympy-editor": 1, "expr": "x", "session": {"history": [evil_step], "index": 0}})

err = raises(lambda: Document(0).open_text(evil_file), ValueError)
check("a history step that calls __import__ is refused", err is not None, err)
err = raises(lambda: Document(0).open_text(f"__import__('os').system('true')"), ValueError)
check("a plain line of 'source' that imports is refused", err is not None, err)
err = raises(lambda: Document(0).open_text(json.dumps({"expr": "MatrixSymbol('M', 2, 2).T"})), ValueError)
check("attributes are refused in a file", err is not None)
check("... and nothing ran", not marker.exists())

# symbols, and an add-on's saved state, are data too
evil_sym = json.dumps({"sympy-editor": 1, "expr": "x", "session": {"history": ["Symbol('x')"], "index": 0,
                        "symbols": [evil_step]}})
raises(lambda: Document(0).open_text(evil_sym))
check("a 'symbol' that runs code does not run", not marker.exists())

# the program's own strings are still the program's
check("Document(\"MatrixSymbol('M', 2, 2).T\") from Python still works",
      Document("MatrixSymbol('M', 2, 2).T").expr == MatrixSymbol("M", 2, 2).T)
check("a plain line of source still opens", Document(0).open_text("x**2 + 1") == x**2 + 1)
check("Matrix(...) given as text opens", Document("Matrix([[1, 2]])").expr == ImmutableMatrix([[1, 2]]))
check('{"expr": 0} opens (it is not "no expression")', Document(y).open_text('{"expr": 0}') == 0)
d = Document(x); d.open_text(json.dumps({"sympy-editor": 1, "expr": "x",
                                         "session": {"history": ["Symbol('x')"], "allow_invalid": "false"}}))
check('"allow_invalid": "false" is read as off', d.allow_invalid is False)

## 2. Versions of the file format

Every file says its format (`"sympy-editor"`) and the oldest reader that can
take it (`"min-reader"`). Older formats are upgraded one step at a time
(`MIGRATIONS`), a newer one opens when its `min-reader` allows, and the rest
is refused by name. `tests/formats/format-1.sympy` is frozen: it must open
for good.

In [ ]:
from sympy_editor import document as D

frozen = (ROOT / "tests" / "formats" / "format-1.sympy").read_text(encoding="utf-8")
d = Document(0)
check("the frozen format-1 file opens", d.open_text(frozen) == x**2 + 1)
check("... with its history, labels and declared names",
      [str(e) for e in d._history] == ["x**2", "x**2 + 1", "x**2 + 1"] and d.declared["t"].is_positive)

saved = json.loads(Document(x + 1).save_text())
check("a saved file carries its format and min-reader",
      saved["sympy-editor"] == D.SAVE_FORMAT and saved["min-reader"] == D.SAVE_MIN_READER)
check("... and its session carries the format too", saved["session"]["format"] == D.SAVE_FORMAT)

newer = {"sympy-editor": D.SAVE_FORMAT + 1, "min-reader": D.SAVE_FORMAT, "expr": "x + 1",
         "session": {"history": ["x + 1"], "index": 0, "a new field": 1}}
check("a newer file that says this reader can read it opens", Document(0).open_text(json.dumps(newer)) == x + 1)
newer["min-reader"] = D.SAVE_FORMAT + 1
err = raises(lambda: Document(0).open_text(json.dumps(newer)), ValueError)
check("a newer file that needs a newer reader is refused by name", err and "newer version" in str(err), err)
for bad in ("two", 0, 1.5, True):
    check(f"format {bad!r} is refused", raises(lambda: Document(0).open_text(json.dumps({"sympy-editor": bad, "expr": "x"})), ValueError))

# a migration, shown on a copy of the registry (nothing is changed for good)
saved_state = (D.SAVE_FORMAT, D.SAVE_MIN_READER, dict(D.MIGRATIONS))
try:
    D.SAVE_FORMAT, D.SAVE_MIN_READER, D.MIGRATIONS = 2, 2, {}
    @D.migration(1)
    def _shout(data):
        data["session"]["labels"] = [l.upper() if l else l for l in data["session"].get("labels", [])]
        return data
    d = Document(0); d.open_text(frozen)
    check("a format-1 file goes through the upgrade to format 2", d._labels[1] == "EDIT: X**2 → X**2 + 1", d._labels)
finally:
    D.SAVE_FORMAT, D.SAVE_MIN_READER, D.MIGRATIONS = saved_state

## 3. Sessions come back as they were saved

In [ ]:
d = Document(Rational(-3, 4)); d.handle({"action": "wrap", "path": "/", "func": "sqrt"})
back = Document("0", **d.export()).expr
check("an unevaluated sqrt(-3/4) comes back unevaluated", back == d.expr and srepr(back) == srepr(d.expr), back)
d2 = Document(0); d2.open_text(d.save_text())
check("... through a saved file too", srepr(d2.expr) == srepr(d.expr), d2.expr)

d = Document("x + 1")
d.handle({"action": "retype", "name": "x", "type": "Symbol", "assumptions": ["positive"]})
d.handle({"action": "undo"})
d.handle({"action": "set", "src": "sqrt(x**2)"})
check("undo undoes a retype (sqrt(x**2) is not x again)", d.expr != Symbol("x"), d.expr)

bad = json.dumps({"sympy-editor": 1, "session": {"history": ["Transpose(Symbol('x'))"]}})
d = Document(y)
snap = d.handle({"action": "openfile", "text": bad})
check("a file with a step that cannot be drawn is refused", snap.get("error"), snap.get("error"))
check("... and the document still answers", d.handle({"action": "snapshot"})["src"] == "y")

from sympy_editor.addons import Addon
class Keeper(Addon):
    name = "keeper"
    def export_state(self, doc): return {"kept": True}
d = Document(x, addons=[], available=[Keeper()], addon_state={"keeper": {"kept": True}})
check("an add-on that is off keeps its saved state in the export", (d.export().get("addon_state") or {}).get("keeper"))

## 4. A click edits what was clicked (the view tree)

SymPy draws some pieces in another order than it stores them. Each drawn
piece must carry its own path.

In [ ]:
import re

def drawn_paths(expr):
    # the paths in the order the formula draws them
    return re.findall(r"path=([^}]*)", Document(expr).snapshot()["latex"])

f = Function("f")
d = Document(Integral(x*y, (x, 0, y), (y, 0, 1)))
first_limit = drawn_paths(d.expr)[1]                 # the first thing drawn after the whole: the outer lower limit
d.replace(first_limit, "5")
check("editing the outer lower limit of a double integral changes the outer one",
      d.expr == Integral(x*y, (x, 0, y), (y, 5, 1)), d.expr)

d = Document(Derivative(f(x, y), (x, 2), (y, 2)))
paths = drawn_paths(d.expr)
check("Derivative: every drawn piece has a path of its own", len(paths) == len(set(paths)), paths)

d = Document(Tuple(1, Tuple(2, 1)))
d.replace(drawn_paths(d.expr)[1], "9")
check("nested tuples: the first 1 drawn is the first 1 edited", d.expr == Tuple(9, Tuple(2, 1)), d.expr)

sp = ImmutableSparseMatrix([[x, 0], [0, 1]])
d = Document(sp)
check("sparse matrix: no key or shape is drawn with a path", all(p.endswith("/1") or p == "/" for p in drawn_paths(sp)),
      drawn_paths(sp))

In [ ]:
# 👆 by hand
SympyEditorWidget(Integral(x*y, (x, 0, y), (y, 0, 1)) + Derivative(f(x, y), (x, 2), (y, 2)))

**Steps.** Click the lower limit `0` of the *outer* (left) integral and type `5`, Enter.
Then click the exponent `2` over `∂y` and type `3`.

**Expect.** The outer integral becomes `∫₅¹`, the inner one keeps `∫₀^y`;
the derivative becomes `∂⁵/∂y³∂x²` — the `y` count changed, not the `x` one.
In the source line under the formula, selecting a limit selects that same limit.

## 5. Operators follow the order you see

In [ ]:
d = Document(x**2 + x)
kids = d.snapshot()["nodes"]
# the front end sends left/right as the arguments drawn left and right of the operator
left, right = list(d.expr.args).index(x**2), list(d.expr.args).index(x)
d.handle({"action": "operator", "path": "/", "left": left, "right": right, "op": "/"})
check("x**2 + x with / at the + gives x (x**2/x), not 1/x", d.expr == x, d.expr)

A, B = MatrixSymbol("A", 2, 2), MatrixSymbol("B", 2, 2)
d = Document(A - B)
a_i = [i for i, a in enumerate(d.expr.args) if a == A][0]
b_i = 1 - a_i
d.handle({"action": "operator", "path": "/", "left": a_i, "right": b_i, "op": "*"})
check("A - B with * gives A*B (not B*A)", d.expr == A*B, d.expr)

In [ ]:
# 👆 by hand
SympyEditorWidget(x**2*y*sin(x)*cos(x) + (x + y)/z)

**Steps.** (a) Click the `·` between `y` and `sin(x)` and type `+`.
(b) Undo. Click the `+` in `(x + y)/z`, press ↓, type `5`.

**Expect.** (a) `x²y + sin(x)cos(x)` — split where you clicked.
(b) The caret goes right after the `+`, on the numerator's line: `(x + 5y)/z`, not `(x+y)/(5z)`.

## 6. The Python script rebuilds every step

In [ ]:
d = Document(x + 1)
d.handle({"action": "set", "src": "Limit(sin(x)/x, x, 0)"})
d.handle({"action": "set", "src": "Integral(_1, _2)"})
d.handle({"action": "wrap", "path": "/", "func": "sqrt"})
d.handle({"action": "retype", "name": "x", "type": "Symbol", "assumptions": ["positive"]})
d.handle({"action": "set", "src": "x + steps + Symbol('lambda')"})
script = d.python_script("verify")
space = {}
try:
    exec(script, space)
    ran = True
except Exception as exc:
    ran = False
    print(script)
check("the exported script runs (Limit, placeholders, odd names, a retype)", ran)

## 7. Empty slots, Tab, the keep chooser, ranges

In [ ]:
d = Document("Matrix([[_1, _2, _3], [_4, _5, _6], [_7, _8, _9], [_10, _11, _12]])")
names = [d.snapshot()["nodes"][p]["src"] for p in d.snapshot()["placeholders"]]
check("Tab order of empty slots is reading order", names == [f"_{i}" for i in range(1, 13)], names)
d = Document("_1/_2")
names = [d.snapshot()["nodes"][p]["src"] for p in d.snapshot()["placeholders"]]
check("a fraction's numerator slot comes before the denominator", names == ["_1", "_2"], names)

In [ ]:
# 👆 by hand
SympyEditorWidget(x**2 + y)

**Steps.** (a) Type `\frac` in the source line and Enter, then press Tab twice.
(b) Undo back to `x² + y`. Click `x`, press ↑ (the power is selected), press Backspace:
the *Keep* chooser opens with `x` focused. Press **Backspace** again (or ↑).

**Expect.** (a) Tab goes numerator, then denominator.
(b) The second Backspace keeps `x`: the formula is `x + y`. (It used to reopen the chooser.)

In [ ]:
# 👆 by hand
SympyEditorWidget(And(x > 1, y < 2, z > 0))

**Steps.** Select `x > 1`, then Shift+→ to take `y < 2` into the range. Enter, change `2` to `7`, Enter.
Then select the `∧` between two terms and use the toolbar's ↑, ← and → buttons.

**Expect.** The range edits (no "Could not parse"). With the operator selected, ↑ selects the whole
`And`, ← and → select the term on that side — exactly what the keys do.

## 8. Moving around a matrix, the caret and the source line

In [ ]:
# 👆 by hand
SympyEditorWidget(Tuple(Matrix([[x, y + 1], [z, 1]]), w))

**Steps.** (a) Click `y` inside `y + 1`, press ←. (b) Select the cell `y + 1`, press → twice.
(c) Put the caret just left of a `+` in the source line (e.g. after `y`), then look at the formula.

**Expect.** (a) `x` is selected (same row), not `z`. (b) → from the last cell of a row steps *out*
of the matrix (to what follows it), never down to the next row.
(c) The formula caret sits at the end of `y`, and a caret placed in the formula at the end of a term
shows in the source line right after that term (`y| + 1`), not after the `+`.

In [ ]:
# 👆 by hand — the empty view
SympyEditorWidget(x + y)

**Steps.** Select the whole expression and press Delete: the view empties. Type `((` and Enter.

**Expect.** An error is shown, but the field stays with `((` in it and keeps the focus: fix it
(`(x)`) and Enter works. Changing the selection afterwards clears the error line.

## 9. Sessions in a notebook, and files next to it

The widget keeps what the editor keeps (sessions, add-on switches, zoom) in
the kernel's store — the same folder `serve()` uses — not in the browser.
Here it uses a scratch folder so your real sessions are left alone.

In [ ]:
store = SCRATCH / "store"
wk = SympyEditorWidget(x**2 + 1, store=store, save_dir=SCRATCH, options={"sessions": True, "rememberZoom": True})
sent = []
real_send = wk.send

# keep: written by the kernel, answered as a message of its own (the trait only ever holds a snapshot)
wk.send = lambda content, buffers=None: sent.append(content)
before = wk.snapshot
wk._on_msg(wk, {"action": "keep", "key": "zoom", "value": "1.5", "_req": 1}, [])
check("keep writes into the kernel's store", (store / "zoom.json").read_text() == "1.5")
wk._on_msg(wk, {"action": "preview", "src": "z**2", "_req": 2}, []); wk.wait(10)
check("a preview does not go into the shared snapshot trait", wk.snapshot == before and sent[-1].get("preview"))

p = wk.save_formula()
check("save_formula writes a .sympy next to the notebook", p.exists() and p.suffix == ".sympy", p)
p2 = wk.save_formula()
check("... never over a file that is there", p2 != p, (p, p2))
wk.expr = cos(x)
check("open_formula takes it back", wk.open_formula(p) == x**2 + 1)
wk.send = real_send
wk

**Steps (the widget above).** Open ≡ → make a *new session*, type `sin(t)`; switch back to the first
session; then re-run the cell (a fresh widget on the same store).
Also ≡ → **File → Save formula…**.

**Expect.** Both sessions are listed after re-running (they are in the store, not the browser), and
switching between them keeps each history. *Save formula* says `saved: …/formula….sympy` —
a file in the scratch folder, not a browser download.

## 10. The server's store

In [ ]:
from sympy_editor.store import Store
import threading

check("store=False answers with an error, so the page keeps its own copy",
      "error" in Store(False).answer({"action": "keep", "key": "zoom", "value": "2"}))

s = Store(SCRATCH / "parallel")
errors = []
def write(i):
    r = s.answer({"action": "keep", "key": "sessions", "value": str(i)})
    if "error" in r: errors.append(r)
threads = [threading.Thread(target=write, args=(i,)) for i in range(16)]
[t_.start() for t_ in threads]; [t_.join() for t_ in threads]
check("16 simultaneous writes of one name all land", not errors, errors[:1])
check("... and no temporary file is left behind", not list((SCRATCH / "parallel").glob("*.new")))

from sympy_editor.server import EditorServer
srv = EditorServer(Document(x), port=0, host="127.0.0.2", store=False)
check("a server on another loopback address still checks the Host header", not srv.accepts_host("evil.example"))
srv.server_close()

## 11. Add-ons

In [ ]:
try:
    from sympy_editor_tree import ADDON as TREE
    d = Document(x + f(y), addons=[TREE])
    args = list(d.expr.args)
    d.handle({"action": "addon", "addon": "tree", "method": "move", "from": [args.index(x)], "to": [args.index(f(y))]})
    check("tree: x moved into f(y) gives f(y, x)", d.expr == f(y, x), d.expr)
except ImportError as exc:
    print("tree add-on not available:", exc)

try:
    from sympy_editor_latex import ADDON as LATEX
    x1, lam = Symbol("x_1", positive=True), Symbol("lamda")
    d = Document(x1 + lam, addons=[LATEX])
    got = LATEX.read(d, {"latex": r"x_1 + \lambda"})
    check("LaTeX: x_1 and \\lambda are the document's own symbols", got["ok"] and got["expr"] == x1 + lam, got.get("src"))
    got = LATEX.read(d, {"latex": r"\frac{1}{d x}"})
    check("LaTeX: \\frac{1}{d x} reads as 1/(d*x)", got["ok"] and got["expr"] == 1 / (Symbol("d") * x), got.get("src"))
except ImportError as exc:
    print("LaTeX add-on not available:", exc)

try:
    from sympy_editor_plot import ADDON as PLOT
    d = Document(sin(x), addons=[PLOT])
    for n in (1, 6000):
        snap = d.handle({"action": "addon", "addon": "plot", "method": "samples", "path": "/", "span": [0, 1], "n": n})
        res = (snap.get("query") or {}).get("result") or {}
        xs = res.get("x") or []
        ys = (res.get("curves") or [{}])[0].get("y") or []
        check(f"plot: n={n} gives as many x as y, without an error",
              not (snap.get("query") or {}).get("error") and len(xs) == len(ys) and len(xs) >= 2,
              (snap.get("query") or {}).get("error") or (len(xs), len(ys)))
except ImportError as exc:
    print("plot add-on not available:", exc)

try:
    from sympy_editor_handwriting import HandwritingAddon
    class Reads:                                   # a stand-in recognizer: no model is needed here
        def __init__(self, latex): self.latex = latex
        def status(self): return {"available": True}
        def warm(self, background=True): return True
        def recognize(self, strokes, beam=4, limit=5, context=None):
            return {"candidates": [{"latex": self.latex}], "ms": 1.0, "stand_in": "\\ctx"}
    for piece in (f(x), Symbol("xy"), Symbol("x_1"), I + 1):
        d = Document(piece, addons=[HandwritingAddon(Reads(r"\ctx + 1"))])
        snap = d.handle({"action": "addon", "addon": "handwriting", "method": "write",
                         "strokes": [[[0, 30, 0], [20, 30, 5]]], "context": [0, 0, 20, 20], "nest": "/", "beam": 1})
        best = snap["query"]["result"]["candidates"][0]
        d.handle({"action": "addon", "addon": "handwriting", "method": "insert", "latex": best["latex"],
                  "path": "/", "nest": best.get("nest"), "display": best["display"]})
        check(f"handwriting around {piece} keeps it as it is", d.expr == piece + 1, d.expr)
except ImportError as exc:
    print("handwriting add-on not available:", exc)

try:
    from sympy_editor_matching import MatchingAddon
    marker2 = SCRATCH / "rule-ran"
    evil_rule = f"__import__('pathlib').Path({str(marker2)!r}).write_text('x') -> 1"
    d = Document(x, addons=[MatchingAddon()], addon_state={"matching": {"rules": ["sin(a_)**2 -> 1 - cos(a_)**2", evil_rule]}})
    check("rewrite rules from saved state are read, never run", not marker2.exists())
    check("... and the good rule is kept", [str(r.pattern) for r in d.addons["matching"].rules(d)] == ["sin(a_)**2"])
except ImportError as exc:
    print("matching add-on not available:", exc)

In [ ]:
# 👆 by hand — the LaTeX field in the formula, and the tree panel switched off and on
SympyEditorWidget(Symbol("b")**Symbol("i") + 1, addons=["sympy_editor_latex", "sympy_editor_tree"])

**Steps.** Select the exponent `i`, press the **LaTeX** tool, type `\cos p`, apply.
Open the field again and tap *inside* it several times.
Then ≡ → Add-ons: switch the tree off and on a few times.

**Expect.** `b^{cos(p)} + 1`; while typing, the `i` is hidden and the field stands in its place;
taps in the field never select anything behind it. Switching the tree off and on leaves no stray
behaviour (each switch removes its page listener).

## 12. On a phone or a touch screen (by hand)

These are for the apps or a browser with a touch screen; there is no widget to run.

| Check | Expect |
|---|---|
| Hold a finger on a term | it is selected, and the app gives a short vibration |
| Android **Back** with the drawer / help / history / LaTeX field open | each press closes one thing; with nothing open the app goes to the background |
| Copy in the editor, paste in another app (and back) | the system clipboard both ways |
| History → Save ▾ → *print or PDF*, or ≡ → *Print history…* | the system print dialog, which can save a PDF |
| Tap a `.sympy` file in a file manager or mail | it opens in the app, in a session of its own |
| Rotate / split-screen while the save dialog is open, then save | the file is written (not empty) |
| Mac: File → New Window, edit in both | both windows edit; Interrupt in one stops only that one |
| Mac: edit, then quit at once | the edit is in the session when the app opens again |
| A mouse pressed on the formula and released outside, then a finger tap | the tap selects (it is not taken for a pinch) |

In [ ]:
passed = sum(ok for _, ok in RESULTS)
print(f"{passed} of {len(RESULTS)} checks pass")
for what, ok in RESULTS:
    if not ok:
        print("✗", what)